# 3_X_RAG_Retrieval_Augmented_Generation_Claudio

**Notebook de estudio: RAG (Retrieval Augmented Generation)**

Este documento explica cómo conectar LLMs con fuentes de datos externas para mejorar la precisión y contextualización de las respuestas.

---

# PARTE 1: AJUSTE DE MODELOS - CONTEXTO

## 1.1 ¿Por qué Ajustar los Modelos?

Los **modelos fundacionales** destacan por su flexibilidad y capacidad para comprender el lenguaje humano. Sin embargo, cuando se enfrentan a **tareas muy específicas**, pueden presentar limitaciones:

- No ofrecen resultados completos para dominios especializados
- Depende de los datos con los que fueron entrenados
- Pueden "estancarse" en ciertas funciones particulares

### Tres Estrategias para Adaptar Modelos

```
┌─────────────────────────────────────────────────────────────────┐
│                    ESPECTRO DE TÉCNICAS                        │
├─────────────────────────────────────────────────────────────────┤
│  PROMPT ENGINEERING → RAG → FINE-TUNING                        │
│                                                                 │
│  Menor dificultad        ←────────→        Mayor dificultad    │
│  Menos recursos          ←────────→        Más recursos        │
│  Menor versatilidad      ←────────→        Mayor versatilidad  │
└─────────────────────────────────────────────────────────────────┘
```

## 1.2 Prompt Engineering

### Definición

Diseñar y modificar el texto de entrada al modelo para orientarlo, a través de contexto, indicaciones o ejemplos, con el fin de obtener una respuesta precisa y adecuada.

### Ejemplo Práctico

**Sin contexto:**
```
Pregunta: ¿Qué es un girasol?
Respuesta: Un girasol es una planta que destaca por su gran flor amarilla.
```

**Con contexto (Prompt Engineering):**
```
Contexto: Estás en un taller de jardinería y el instructor pregunta sobre plantas comunes.
Pregunta: ¿Qué es un girasol?
Respuesta: El girasol es una planta muy apreciada tanto por su belleza como por sus semillas, 
que se utilizan en la alimentación y para extraer aceite.
```

**La diferencia:** El mismo modelo, la misma pregunta, pero el contexto cambia completamente el enfoque de la respuesta.

## 1.3 Fine-Tuning (Ajuste Fino)

### Definición

El ajuste fino consiste en **modificar y optimizar los parámetros de un modelo ya entrenado**. Normalmente, se reentrenan las capas superiores de la red neuronal utilizando nuevos datos correctamente etiquetados.

### ¿Cuándo usar Fine-Tuning?

| Situación | Ejemplo |
|-----------|---------|
| Tareas muy especializadas | LLM experto en normativas bancarias |
| Adaptación a múltiples contextos | Chatbot para atención al cliente |
| Dominios específicos con vocabulario técnico | Medicina, derecho, ingeniería |

### Características

- ✅ Mayor versatilidad y especialización
- ✅ El modelo "aprende" permanentemente los nuevos conocimientos
- ❌ Requiere más recursos computacionales
- ❌ Proceso costoso y lento
- ❌ No es eficiente si los datos cambian frecuentemente

## 1.4 Comparativa de Estrategias

| Aspecto | Prompt Engineering | RAG | Fine-Tuning |
|---------|-------------------|-----|-------------|
| **Dificultad** | Baja | Media | Alta |
| **Recursos** | Mínimos | Moderados | Elevados |
| **Versatilidad** | Limitada | Alta | Muy alta |
| **Actualización de datos** | Inmediata | Inmediata | Requiere reentreno |
| **Trazabilidad** | N/A | Sí (fuentes) | No |
| **Coste** | Bajo | Medio | Alto |

### Principio Clave

> **No hay una estrategia única que funcione mejor en todos los casos.** La elección depende de las particularidades del caso de uso. Muchas veces, las técnicas pueden **combinarse** para obtener mejores resultados.

---

# PARTE 2: INTRODUCCIÓN A RAG

## 2.1 ¿Qué es RAG?

**RAG = Retrieval Augmented Generation** (Generación Aumentada por Recuperación)

Es una arquitectura que permite a un LLM **buscar respuestas empleando un contexto privado de documentación**, en lugar de depender únicamente de su conocimiento preentrenado.

### Origen

Creada por **Meta (Facebook)** en 2020, presentada en el artículo "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks".

### ¿Por qué no usar solo Fine-Tuning?

| Problema del Fine-Tuning | Solución con RAG |
|--------------------------|------------------|
| Proceso costoso y lento | Búsqueda en tiempo real |
| No eficiente si los datos cambian frecuentemente | Acceso a datos actualizados instantáneamente |
| No se puede rastrear de dónde obtuvo la información | Trazabilidad completa de fuentes |
| Cualquiera con acceso al modelo puede consultar toda la información | Control de acceso granular |
| Riesgo de alucinaciones | Respuestas fundamentadas en documentos reales |

## 2.2 El Proceso de RAG: Dos Etapas

```
┌─────────────────────────────────────────────────────────────────┐
│                    ARQUITECTURA RAG                            │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌─────────────────┐         ┌─────────────────┐              │
│   │   RETRIEVAL     │         │   GENERATION    │              │
│   │  (Recuperación) │   →     │  (Generación)   │              │
│   └─────────────────┘         └─────────────────┘              │
│                                                                 │
│   1. Motor de búsqueda          2. Modelo generativo (LLM)     │
│      obtiene fragmentos            usa la información          │
│      relevantes de la base         recuperada para             │
│      de datos o corpus vectorial   producir respuestas         │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

### Etapa 1: Retrieval (Recuperación)

Se emplea un motor de búsqueda para obtener fragmentos relevantes de una base de datos o corpus vectorial.

### Etapa 2: Generation (Generación)

Un modelo generativo (típicamente un LLM) utiliza la información recuperada para producir respuestas.

## 2.3 Flujo Completo del Proceso RAG

```
PASO 1: Entrada del usuario
        ↓
        El usuario envía una consulta o prompt

PASO 2: Conversión a embedding
        ↓
        La consulta se convierte en un vector (embedding)
        usando técnicas de representación vectorial

PASO 3: Búsqueda en base de datos vectorial
        ↓
        Se busca usando similitud del coseno (u otras métricas)
        para encontrar los fragmentos más cercanos

PASO 4: Integración de información
        ↓
        Se eligen los top-N documentos con mayor similitud
        y se integran al prompt original como contexto

PASO 5: Generación de respuesta
        ↓
        El LLM crea una respuesta enriquecida usando
        tanto la consulta como la información externa

PASO 6: Trazabilidad de referencias
        ↓
        Se incluyen referencias a los documentos utilizados
        para fundamentar la respuesta
```

---

# PARTE 3: CHUNKING (DIVISIÓN EN FRAGMENTOS)

## 3.1 ¿Qué es el Chunking?

El **chunking** es la técnica de **dividir grandes volúmenes de texto o datos en fragmentos más pequeños y manejables** antes de que sean utilizados en el modelo de RAG.

### Analogía

Es como dividir un libro en capítulos o secciones. En lugar de buscar en todo el libro, buscas en el capítulo relevante.

## 3.2 ¿Por qué es Importante?

### 1. Mejora de la Eficiencia

- Los modelos de RAG manejan grandes volúmenes de texto (artículos, libros, bases de datos)
- Alimentar el modelo con documentos enteros sería **ineficiente**
- Los fragmentos pequeños hacen la búsqueda **más rápida y precisa**

### 2. Precisión en la Recuperación

- Al dividir el texto, el modelo puede recuperar piezas de información **más específicas y relevantes**
- Especialmente útil cuando la información es muy detallada o variada
- El modelo identifica exactamente cuál chunk es el más adecuado para una pregunta dada

## 3.3 Estrategias de Chunking

| Estrategia | Descripción | Cuándo usarla |
|------------|-------------|---------------|
| **Por párrafos** | Divide en párrafos completos | Texto narrativo, artículos |
| **Por tamaño fijo** | Fragmentos de N caracteres/tokens | Documentos técnicos |
| **Por oraciones** | Divide en oraciones individuales | Respuestas cortas y precisas |
| **Por semántica** | Divide por temas o significado | Documentos con múltiples temas |
| **Con solapamiento** | Fragmentos se superponen parcialmente | Mantener contexto entre chunks |

In [ ]:
# Ejemplo práctico de chunking
def chunk_text(text, chunk_size=500, overlap=50):
    """
    Divide un texto en chunks de tamaño fijo con solapamiento.
    
    Args:
        text: Texto completo a dividir
        chunk_size: Tamaño de cada chunk en caracteres
        overlap: Caracteres de solapamiento entre chunks
    """
    chunks = []
    start = 0
    
    while start < len(text):
        # Extraer chunk
        end = start + chunk_size
        chunk = text[start:end]
        
        # Ajustar al final de oración si es posible
        if end < len(text):
            last_period = chunk.rfind('.')
            if last_period > 0:
                chunk = chunk[:last_period + 1]
        
        chunks.append(chunk.strip())
        
        # Avanzar con solapamiento
        start += chunk_size - overlap
    
    return chunks

# Ejemplo de uso
texto_ejemplo = """
El Retrieval Augmented Generation (RAG) es una arquitectura que combina
la recuperación de información con la generación de texto. Esta técnica
permite a los modelos de lenguaje acceder a información externa y actualizada,
mejorando significativamente la precisión de sus respuestas. El proceso
involve varios pasos: chunking, embedding, indexación y búsqueda.
"""

chunks = chunk_text(texto_ejemplo, chunk_size=150, overlap=20)

print(f"Número de chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i}: {chunk[:100]}...\n")

---

# PARTE 4: INDEXACIÓN VECTORIAL

## 4.1 ¿Qué es la Indexación Vectorial?

La indexación vectorial consiste en **convertir la información de lenguaje natural en representaciones numéricas de alta dimensión** (vectores).

### Proceso

```
Texto (chunk) → Modelo de Embeddings → Vector numérico

Ejemplo:
"El gato come pescado" → [0.23, -0.45, 0.89, 0.12, -0.67, ...] (768 dimensiones)
```

## 4.2 Modelos de Embeddings

| Modelo | Características | Uso típico |
|--------|-----------------|------------|
| **Word2Vec** | Representaciones estáticas | Básico, palabras individuales |
| **GloVe** | Vectores globales | Contexto general |
| **BERT** | Contextualizados, bidireccionales | Comprensión profunda |
| **Sentence-BERT** | Optimizado para oraciones completas | Similaridad semántica |
| **OpenAI Embeddings** | Alta calidad, API fácil | Producción |
| **HuggingFace Models** | Diversos, open source | Flexibilidad |

## 4.3 Bases de Datos Vectoriales

Los vectores generados se almacenan en bases de datos optimizadas para búsqueda vectorial:

| Base de Datos | Tipo | Características |
|---------------|------|-----------------| 
| **FAISS** | Librería (Meta) | Rápida, eficiente, local |
| **Pinecone** | SaaS | Managed, escalable |
| **Weaviate** | Open Source | Vector + semántica |
| **Chroma** | Open Source | Fácil de usar, local |
| **Elasticsearch** | Search Engine | Full-text + vectorial |
| **pgvector** | Extensión PostgreSQL | SQL + vectores |

## 4.4 Metadatos en la Indexación

Además de los vectores, la indexación puede incluir **metadatos** asociados a cada fragmento:

| Tipo de Metadato | Ejemplo | Utilidad |
|------------------|---------|----------|
| **Origen** | Nombre del documento | Trazabilidad |
| **Fecha** | 2024-01-15 | Filtrado temporal |
| **Categoría** | "Legal", "Técnico" | Filtrado por tema |
| **Autor** | "Juan Pérez" | Atribución |
| **Permisos** | Grupos de usuarios | Control de acceso |

**Beneficio:** Permite realizar búsquedas sofisticadas combinando similitud semántica + filtros de metadatos.

In [ ]:
# Ejemplo de creación de embeddings e indexación
from sentence_transformers import SentenceTransformer
import numpy as np

# Cargar modelo de embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# Documentos de ejemplo
documents = [
    "El RAG permite a los LLMs acceder a información externa",
    "Los embeddings convierten texto en vectores numéricos",
    "El chunking divide documentos en fragmentos manejables",
    "Las bases de datos vectoriales almacenan embeddings",
]

# Generar embeddings
embeddings = model.encode(documents)

print(f"Forma de los embeddings: {embeddings.shape}")
print(f"Cada documento se convierte en un vector de {embeddings.shape[1]} dimensiones\n")

# Mostrar los primeros valores del primer embedding
print(f"Primeros 10 valores del primer documento:")
print(embeddings[0][:10])

---

# PARTE 5: BÚSQUEDA DE INFORMACIÓN

## 5.1 Fases de la Búsqueda

### Fase 1: Consulta

El usuario formula una consulta en lenguaje natural, que puede contener:
- Palabras clave
- Preguntas específicas
- Descripciones vagas del tema

### Fase 2: Conversión de la Consulta en Vector

La consulta se convierte en un vector de características usando el **mismo modelo de embedding** que se usó para indexar los documentos.

```
Consulta: "¿Cómo funciona RAG?"
   ↓
Vector consulta: [0.15, -0.32, 0.78, ...]
```

### Fase 3: Búsqueda en el Índice Vectorial

Se buscan los vectores más cercanos a la consulta usando métricas de distancia.

## 5.2 Métricas de Similitud

### 1. Similitud del Coseno (Más común)

Mide el **ángulo entre dos vectores**, útil para ver qué tan similares son en términos de dirección, sin tener en cuenta su magnitud.

**Fórmula:** cos(θ) = (A · B) / (||A|| × ||B||)

- Valor 1: Vectores idénticos (misma dirección)
- Valor 0: Vectores ortogonales (sin relación)
- Valor -1: Vectores opuestos

### 2. Distancia Euclidiana

Mide la **distancia lineal** entre dos puntos en el espacio multidimensional.

**Fórmula:** d = √Σ(Ai - Bi)²

- Menor distancia = mayor similitud
- Mayor distancia = menor similitud

### Comparativa

| Métrica | Mide | Mejor para |
|---------|------|------------|
| **Coseno** | Ángulo entre vectores | Dirección semántica |
| **Euclidiana** | Distancia absoluta | Magnitud y dirección |

## 5.3 Recuperación y Filtrado

### Recuperación de Fragmentos Relevantes

El sistema recupera los **top-N chunks** más cercanos a la consulta. Normalmente se recuperan varios fragmentos para generar una respuesta completa.

### Filtrado y Refinamiento (Opcional)

Después de la búsqueda inicial, se puede refinar usando metadatos:

- **Filtrado temporal:** Solo documentos de los últimos meses
- **Filtrado por categoría:** Solo documentos "Legales"
- **Filtrado por autor:** Solo documentos de cierto departamento
- **Ajuste de relevancia:** Umbral mínimo de similitud

In [ ]:
# Ejemplo de búsqueda por similitud del coseno
from sklearn.metrics.pairwise import cosine_similarity

def buscar_documentos(consulta, documentos, embeddings, top_k=3):
    """
    Busca los documentos más similares a una consulta.
    
    Args:
        consulta: Texto de búsqueda
        documentos: Lista de documentos originales
        embeddings: Matriz de embeddings de documentos
        top_k: Número de resultados a retornar
    """
    # Convertir consulta a embedding
    consulta_embedding = model.encode([consulta])
    
    # Calcular similitudes
    similitudes = cosine_similarity(consulta_embedding, embeddings)[0]
    
    # Obtener índices de los top_k más similares
    indices_top = np.argsort(similitudes)[::-1][:top_k]
    
    # Preparar resultados
    resultados = []
    for idx in indices_top:
        resultados.append({
            'documento': documentos[idx],
            'similitud': float(similitudes[idx]),
            'indice': idx
        })
    
    return resultados

# Ejemplo de búsqueda
consulta = "¿Qué es el chunking?"
resultados = buscar_documentos(consulta, documents, embeddings, top_k=2)

print(f"Consulta: '{consulta}'\n")
print("Resultados más relevantes:")
for i, res in enumerate(resultados, 1):
    print(f"\n{i}. Similitud: {res['similitud']:.4f}")
    print(f"   Documento: {res['documento']}")

---

# PARTE 6: GENERACIÓN DE CONTENIDO

## 6.1 El Proceso de Generación

Una vez recuperada la información relevante, el modelo generativo (GPT, T5, etc.) produce la respuesta final.

### Entrada al Modelo Generativo

La información recuperada se introduce junto con la consulta original:

```
Prompt para el LLM:
─────────────────────────────────────────
Contexto relevante recuperado:
[Chunk 1]: El RAG permite a los LLMs...
[Chunk 2]: Los embeddings convierten...

Consulta del usuario: ¿Cómo funciona RAG?

Instrucción: Responde usando solo la información del contexto proporcionado.
─────────────────────────────────────────
```

## 6.2 Contextualización y Fusión

El modelo debe:

1. **Fusionar información** de múltiples chunks
2. **Elaborar** una respuesta coherente alineada con el contexto
3. **Generar** texto fluido que no parezca fragmentos desconectados

## 6.3 Características de la Respuesta Generada

| Característica | Descripción |
|----------------|-------------|
| **Precisa** | Basada en documentos reales, no alucinaciones |
| **Contextualizada** | Adaptada a la consulta específica |
| **Fundamentada** | Con referencias a las fuentes utilizadas |
| **Coherente** | Texto fluido, no copia-pega de fragmentos |

---

# PARTE 7: EVALUACIÓN DEL MODELO RAG

## 7.1 Métricas de Recuperación (Retrieval)

### Precisión (Precision)

Mide el **porcentaje de fragmentos recuperados que son realmente relevantes**.

**Fórmula:** Precision = (Fragmentos relevantes recuperados) / (Total fragmentos recuperados)

**Ejemplo:** Si el modelo recupera 10 fragmentos y solo 6 son relevantes → Precision = 60%

### Recall (Cobertura)

Mide el **porcentaje de fragmentos relevantes que han sido recuperados**.

**Fórmula:** Recall = (Fragmentos relevantes recuperados) / (Total fragmentos relevantes existentes)

**Ejemplo:** Si hay 10 fragmentos relevantes en total y el modelo recupera 6 → Recall = 60%

### Relación Precision vs Recall

```
Alta Precisión + Alto Recall = Sistema ideal
Alta Precisión + Bajo Recall  = Recupera pocos, pero buenos
Baja Precisión + Alto Recall  = Recupera muchos, con ruido
Baja Precisión + Bajo Recall  = Sistema deficiente
```

## 7.2 Métricas de Generación

| Métrica | Mide |
|---------|------|
| **Faithfulness** | ¿La respuesta es fiel a los documentos fuente? |
| **Answer Relevance** | ¿La respuesta responde a la pregunta? |
| **Context Precision** | ¿El contexto recuperado es relevante? |
| **Context Recall** | ¿Se recuperó todo el contexto necesario? |

---

# PARTE 8: SEGURIDAD EN RAG

## 8.1 El Problema del Control de Acceso

En RAG existe un **desacople** entre:
- El acceso al **índice vectorial**
- El acceso a los **documentos originales**

**Riesgo:** Un usuario sin permiso para consultar ciertos documentos podría obtener información sobre ellos indirectamente a través de las respuestas del LLM.

## 8.2 Soluciones

### Opción A: Metadatos de Control de Acceso

Incluir en los metadatos los **grupos de usuarios autorizados** para cada documento:

```
Chunk: "Información confidencial del departamento de RRHH..."
Metadatos: {
    "autorizado_para": ["rrhh", "direccion"],
    "departamento": "RRHH",
    "nivel_confidencial": "alto"
}
```

Durante la búsqueda, aplicar un **filtro** que restrinja los resultados según los grupos del usuario.

### Opción B: Silos de Información

Segmentar la información en **índices independientes** por departamento o unidad:

```
Índice RRHH → Solo usuarios de RRHH
Índice Legal → Solo usuarios del departamento legal
Índice Ventas → Solo usuarios de ventas
Índice General → Todos los usuarios
```

Cada usuario solo puede consultar los índices de su grupo.

## 8.3 Buenas Prácticas de Seguridad

| Práctica | Descripción |
|----------|-------------|
| **Autenticación** | Verificar identidad antes de permitir consultas |
| **Autorización basada en roles** | Controlar qué índices puede consultar cada usuario |
| **Auditoría** | Registrar qué usuarios consultan qué información |
| **Enmascaramiento** | Ocultar información sensible en los chunks |
| **Validación de respuestas** | Revisar que no se filtre información no autorizada |

In [ ]:
# Ejemplo de implementación de control de acceso en RAG
class RAGConControlDeAcceso:
    """
    Sistema RAG con control de acceso basado en roles.
    """
    
    def __init__(self):
        self.usuarios = {
            "juan": {"grupos": ["rrhh", "general"]},
            "maria": {"grupos": ["legal", "general"]},
            "pedro": {"grupos": ["general"]}
        }
        
        # Documentos con metadatos de acceso
        self.documentos = [
            {
                "texto": "Política de vacaciones 2024: 25 días laborables...",
                "grupos": ["rrhh", "general"]
            },
            {
                "texto": "Contrato confidencial con proveedor X...",
                "grupos": ["legal"]
            },
            {
                "texto": "Manual de bienvenida para empleados...",
                "grupos": ["general"]
            }
        ]
    
    def puede_acceder(self, usuario, documento):
        """Verifica si un usuario puede acceder a un documento."""
        grupos_usuario = set(self.usuarios[usuario]["grupos"])
        grupos_doc = set(documento["grupos"])
        return len(grupos_usuario & grupos_doc) > 0
    
    def buscar(self, usuario, consulta):
        """Busca documentos filtrando por permisos del usuario."""
        resultados = []
        for doc in self.documentos:
            if self.puede_acceder(usuario, doc):
                # Aquí iría la lógica de similitud
                if consulta.lower() in doc["texto"].lower():
                    resultados.append(doc["texto"])
        return resultados

# Demostración
rag = RAGConControlDeAcceso()

print("Búsqueda de 'vacaciones' por Juan (RRHH):")
print(rag.buscar("juan", "vacaciones"))

print("\nBúsqueda de 'vacaciones' por Pedro (solo General):")
print(rag.buscar("pedro", "vacaciones"))

print("\nBúsqueda de 'contrato' por María (Legal):")
print(rag.buscar("maria", "contrato"))

---

# RESUMEN FINAL

## Flujo Completo de RAG

```
┌─────────────────────────────────────────────────────────────────┐
│                    PIPELINE RAG COMPLETO                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. PREPARACIÓN                                                │
│     ├── Chunking: Dividir documentos en fragmentos             │
│     ├── Embeddings: Convertir chunks a vectores                │
│     └── Indexación: Almacenar en base de datos vectorial       │
│                              ↓                                 │
│  2. CONSULTA                                                   │
│     ├── Usuario hace una pregunta                              │
│     ├── Consulta → Embedding                                   │
│     └── Búsqueda por similitud (coseno)                        │
│                              ↓                                 │
│  3. RECUPERACIÓN                                               │
│     ├── Filtrado por metadatos (seguridad)                     │
│     └── Selección de top-N chunks                              │
│                              ↓                                 │
│  4. GENERACIÓN                                                 │
│     ├── Chunks + Consulta → Prompt enriquecido                 │
│     ├── LLM genera respuesta                                   │
│     └── Incluye referencias a fuentes                          │
│                              ↓                                 │
│  5. EVALUACIÓN                                                 │
│     ├── Precision: ¿Recuperados son relevantes?                │
│     └── Recall: ¿Se recuperó todo lo relevante?                │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## Conceptos Clave

| Concepto | Definición |
|----------|------------|
| **RAG** | Retrieval Augmented Generation - Generación aumentada con recuperación |
| **Chunking** | Dividir documentos en fragmentos manejables |
| **Embedding** | Representación vectorial de texto |
| **Indexación Vectorial** | Almacenar vectores para búsqueda eficiente |
| **Similitud del Coseno** | Métrica para comparar vectores |
| **Fine-tuning** | Reentrenar modelo con datos específicos |
| **Prompt Engineering** | Diseñar entradas para orientar al modelo |
| **Precision** | % de recuperados que son relevantes |
| **Recall** | % de relevantes que fueron recuperados |

## Comparativa: Prompt Engineering vs RAG vs Fine-tuning

| Aspecto | Prompt Engineering | RAG | Fine-tuning |
|---------|-------------------|-----|-------------|
| Complejidad | ⭐ Baja | ⭐⭐ Media | ⭐⭐⭐ Alta |
| Recursos | ⭐ Mínimos | ⭐⭐ Moderados | ⭐⭐⭐ Elevados |
| Actualización datos | Manual | Automática | Requiere reentreno |
| Trazabilidad | ❌ No | ✅ Sí | ❌ No |
| Conocimiento privado | ❌ No | ✅ Sí | ✅ Sí |
| Especialización | ❌ Baja | ⭐⭐ Media | ⭐⭐⭐ Alta |

## Cuándo Usar Cada Técnica

- **Prompt Engineering**: Cuando necesitas orientar el comportamiento general del modelo sin datos específicos
- **RAG**: Cuando necesitas que el modelo acceda a información privada, actualizada y trazable
- **Fine-tuning**: Cuando necesitas especialización profunda en un dominio específico

---

**Notebook generado para estudio y consulta futura**

Basado en: *3_X_RAG_Retrieval_Augmented_Generation*

# Diccionario de Términos - RAG (Retrieval Augmented Generation)

Glosario de conceptos clave del módulo de RAG.

---

## A

| Término | Definición |
|---------|------------|
| **Actualización Incremental** | Capacidad de añadir nuevos documentos o modificar existentes en el índice vectorial sin necesidad de reindexar todo el corpus. |
| **Answer Relevance** | Métrica que evalúa si la respuesta generada responde directamente a la pregunta formulada por el usuario. |

---

## B

| Término | Definición |
|---------|------------|
| **Base de Datos Vectorial** | Sistema de almacenamiento especializado para vectores de alta dimensión que permite búsquedas por similitud semántica. Ejemplos: FAISS, Pinecone, Weaviate, Chroma. |
| **Búsqueda por Similitud** | Método de recuperación que encuentra los vectores más cercanos a la consulta usando métricas como similitud del coseno o distancia euclidiana. |
| **Búsqueda Semántica** | Tipo de búsqueda que encuentra información relevante basándose en el significado (semántica) en lugar de coincidencias exactas de palabras. |

---

## C

| Término | Definición |
|---------|------------|
| **Chunk** | Fragmento individual de texto resultante del proceso de chunking. Puede ser un párrafo, sección o número fijo de caracteres/tokens. |
| **Chunking** | Técnica de dividir grandes volúmenes de texto en fragmentos más pequeños y manejables (chunks) antes de procesarlos en RAG. Mejora eficiencia y precisión. |
| **Compresión de Contexto** | Técnicas para reducir la longitud de los chunks recuperados y maximizar la información relevante dentro de la ventana de contexto del LLM. |
| **Consulta (Query)** | Pregunta o texto de búsqueda que el usuario formula al sistema RAG para recuperar información relevante. |
| **Consulta Vectorial** | Representación numérica (vector) de la consulta del usuario, generada usando el mismo modelo de embeddings que los documentos. |
| **Context Precision** | Métrica que evalúa si el contexto recuperado es relevante para la consulta realizada. |
| **Context Recall** | Métrica que evalúa si se recuperó todo el contexto necesario para responder la consulta adecuadamente. |
| **Contextualización** | Proceso donde el LLM fusiona e integra la información de múltiples chunks recuperados para construir una respuesta coherente y fluida. |
| **Control de Acceso** | Mecanismos de seguridad que restringen qué información puede consultar cada usuario basándose en roles, grupos o permisos. |
| **Corpus** | Conjunto de textos utilizados para alimentar el sistema RAG. |

---

## D

| Término | Definición |
|---------|------------|
| **Desacople de Acceso** | Problema de seguridad en RAG donde el acceso al índice vectorial está separado del acceso a los documentos originales, permitiendo fugas de información. |
| **Dimensionalidad** | Número de dimensiones de un vector de embedding. Típicamente entre 384 y 1536 dimensiones según el modelo utilizado. |
| **Distancia Euclidiana** | Métrica que mide la distancia lineal entre dos puntos en el espacio vectorial. Menor distancia = mayor similitud. |

---

## E

| Término | Definición |
|---------|------------|
| **Embedding** | Representación vectorial densa de texto que captura su significado semántico. En RAG, tanto documentos como consultas se convierten a embeddings. |
| **Embedding Multimodal** | Representaciones vectoriales que capturan no solo texto sino también imágenes, audio u otros tipos de datos para RAG multimodal. |
| **Espacio Vectorial** | Representación matemática de alta dimensión donde los embeddings de texto se posicionan según su significado semántico. |

---

## F

| Término | Definición |
|---------|------------|
| **FAISS** | Librería de Meta para búsqueda de similitud eficiente en vectores de alta dimensión. Se usa localmente. |
| **Faithfulness** | Métrica de calidad que evalúa si la respuesta generada es fiel a los documentos fuente utilizados en la recuperación. |
| **Filtrado por Metadatos** | Proceso de refinar resultados de búsqueda usando información adicional (fecha, categoría, autor, permisos) asociada a cada chunk. |
| **Filtrado por Roles** | Técnica que incluye grupos de usuarios autorizados en los metadatos de cada chunk y filtra resultados durante la búsqueda según estos permisos. |
| **Fusión de Información** | Capacidad del modelo de combinar datos de varios fragmentos recuperados en una respuesta unificada sin que parezcan datos desconectados. |

---

## G

| Término | Definición |
|---------|------------|
| **Generation (Generación)** | Segunda etapa de RAG: proceso donde un modelo generativo (LLM) utiliza la información recuperada para producir una respuesta coherente y contextualizada. |

---

## H

| Término | Definición |
|---------|------------|
| **Hierarchical Indexing** | Estructura de indexación en niveles (resumen → sección → párrafo) que permite recuperar información a diferentes granularidades. |
| **Híbrido (Sparse + Dense)** | Combinación de búsqueda tradicional por palabras clave (sparse) con búsqueda vectorial semántica (dense) para mejores resultados. |

---

## I

| Término | Definición |
|---------|------------|
| **Indexación Vectorial** | Proceso de convertir texto en representaciones numéricas de alta dimensión (vectores/embeddings) y almacenarlos en una base de datos optimizada para búsqueda vectorial. |
| **Ingesta de Documentos** | Proceso de cargar, procesar e indexar documentos en el sistema RAG para hacerlos disponibles para búsqueda. |

---

## L

| Término | Definición |
|---------|------------|
| **Limpieza de Texto** | Proceso de normalizar y preparar el texto extraído (eliminar ruido, normalizar espacios, corregir codificación) antes del chunking. |

---

## M

| Término | Definición |
|---------|------------|
| **Memoria en RAG** | Capacidad de mantener contexto de conversaciones anteriores para consultas de seguimiento, combinando historial con recuperación de documentos. |
| **Metadatos** | Información adicional asociada a cada chunk: origen del documento, fecha, categoría, autor, nivel de acceso, etc. Facilitan búsquedas sofisticadas. |
| **Modelo de Embeddings** | Modelo especializado en convertir texto a vectores. Ejemplos: Word2Vec, GloVe, BERT, Sentence-BERT, OpenAI Embeddings. |
| **Multi-query Retrieval** | Técnica donde una consulta de usuario se expande en múltiples consultas relacionadas para mejorar la cobertura de recuperación. |

---

## P

| Término | Definición |
|---------|------------|
| **Parent Document Retrieval** | Estrategia donde se recupera el documento completo "padre" a partir del chunk más relevante para proporcionar más contexto al LLM. |
| **Parser de Documentos** | Componente que extrae texto de diferentes formatos de archivo (PDF, Word, HTML, etc.) para su procesamiento en RAG. |
| **Pinecone** | Base de datos vectorial como servicio (SaaS) en la nube, escalable y gestionada. |
| **Pipeline RAG** | Flujo completo de procesamiento: chunking → embedding → indexación → consulta → recuperación → generación → evaluación. |
| **Precisión (Precision)** | Métrica que mide el porcentaje de fragmentos recuperados que son realmente relevantes para la consulta. |
| **Prompt Enriquecido** | Prompt que incluye tanto la consulta original del usuario como los chunks de contexto recuperados, proporcionado al LLM para generación. |
| **pgvector** | Extensión de PostgreSQL que añade capacidades de almacenamiento y búsqueda vectorial al sistema SQL tradicional. |

---

## R

| Término | Definición |
|---------|------------|
| **RAG (Retrieval Augmented Generation)** | Arquitectura que combina recuperación de información con generación de texto. Permite a un LLM buscar respuestas en documentos externos en lugar de depender solo de su conocimiento pre-entrenado. |
| **Recall (Cobertura)** | Métrica que mide el porcentaje de fragmentos relevantes existentes que fueron efectivamente recuperados. |
| **Recuperación de Fragmentos** | Proceso de extraer los chunks o documentos más cercanos a la consulta vectorial según la métrica de similitud utilizada. |
| **Recuperación (Retrieval)** | Primera etapa de RAG: proceso de buscar y recuperar fragmentos relevantes de una base de datos o corpus vectorial usando técnicas de búsqueda semántica. |
| **Referencia de Fuentes** | Inclusión explícita de los documentos originales utilizados para fundamentar una respuesta generada por RAG. |
| **Re-ranking** | Proceso adicional de reordenar resultados recuperados usando un modelo más sofisticado para mejorar la relevancia final. |

---

## S

| Término | Definición |
|---------|------------|
| **Sentence-BERT** | Variante de BERT optimizada para generar embeddings de oraciones completas, ideal para RAG y búsqueda semántica. |
| **Silos de Información** | Estrategia de seguridad que segmenta datos en índices independientes por departamento o grupo, restringiendo el acceso de usuarios solo a índices autorizados. |
| **Similitud del Coseno** | Métrica que mide el ángulo entre dos vectores para determinar qué tan similares son semánticamente. Valor 1 = idénticos, 0 = sin relación. |
| **Solapamiento (Overlap)** | Técnica de chunking donde los fragmentos consecutivos se superponen parcialmente para mantener contexto entre chunks. |

---

## T

| Término | Definición |
|---------|------------|
| **Top-k** | Parámetro que indica cuántos resultados (chunks/documentos) más relevantes se recuperan de la base de datos vectorial. Ejemplo: top-5 = los 5 más similares. |
| **Trazabilidad** | Capacidad de rastrear qué documentos y fragmentos específicos fueron utilizados para generar una respuesta. Permite verificar fuentes. |

---

## U

| Término | Definición |
|---------|------------|
| **Umbral de Similitud** | Valor mínimo de similitud que debe alcanzar un resultado para ser considerado relevante y devuelto al usuario. |

---

## V

| Término | Definición |
|---------|------------|
| **Ventana de Contexto** | Cantidad máxima de tokens que el LLM puede procesar en una sola llamada, limitando cuántos chunks se pueden incluir en el prompt. |

---

## W

| Término | Definición |
|---------|------------|
| **Weaviate** | Base de datos vectorial open source que combina búsqueda vectorial con capacidades semánticas. |
